# Media / User Types / Companies Migration

**Migration 2 of the run order** — run after `geo_migration.ipynb` and **before** `user_migration.ipynb` (users reference media, user_types and companies).

Migrates the three independent legacy tables that everything else hangs off:

| Legacy (Strapi) | New (postcardv2) | Notes |
|---|---|---|
| `files` (upload plugin) | `media` | url, mime, alt, width, height — `formats`, `hash`, `caption`, `provider` are **dropped** |
| `user_types` | `user_types` | upsert on slug; `seed.py` already seeds `member` + `admin` |
| `companies` | `companies` | legacy has no slug (generated) and no contact fields; `icon` media is **dropped** (no column) |

**Prerequisites (in order):** schema migrated (`npm run migrate:deploy`) → `python scripts/seed.py` → `geo_migration.ipynb`. Idempotent — safe to re-run.

In [1]:
import os, re
from pathlib import Path

import requests
import psycopg
from dotenv import load_dotenv

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(ROOT / ".env")

CMS_BASE_URL = os.environ["CMS_BASE_URL"].rstrip("/")
HEADERS = {"Authorization": f"Bearer {os.environ['CMS_API_TOKEN']}"}
DATABASE_URL = os.environ["DATABASE_URL"]


def slugify(text):
    return re.sub(r"[^a-z0-9]+", "-", (text or "").lower()).strip("-") or None


def attrs(item):
    """Entry fields — Strapi v4 nests them under 'attributes', v5 is flat."""
    return item.get("attributes", item)

def rel(obj):
    """Unwrap a populated relation — v4: {'data': {'attributes': {...}}}, v5: flat dict."""
    if isinstance(obj, dict) and "data" in obj:
        obj = obj["data"]
    if not obj:
        return None
    return obj.get("attributes", obj)



def fetch_all(path, params=None):
    """Fetch every page of a Strapi collection endpoint (data/meta envelope)."""
    items, page = [], 1
    while True:
        p = {"pagination[page]": page, "pagination[pageSize]": 100, **(params or {})}
        r = requests.get(f"{CMS_BASE_URL}{path}", headers=HEADERS, params=p, timeout=60)
        r.raise_for_status()
        body = r.json()
        items.extend(body["data"])
        pg = body.get("meta", {}).get("pagination", {})
        if page >= pg.get("pageCount", 1):
            return items
        page += 1


conn = psycopg.connect(DATABASE_URL)
print("connected to:", DATABASE_URL.rsplit("/", 1)[-1])

connected to: development


## 1. Media — `files` → `media`

The upload plugin endpoint `/api/upload/files` returns **flat** objects (no `data/attributes` envelope). Pagination support varies by Strapi version, so the fetch loop guards against a server that ignores the pagination params.

Mapping: `url` (made absolute when relative) → `url`, `mime` → `mime_type`, `alternativeText` (fallback `caption`, then `name`) → `alt`, `width`/`height` as-is.

`media.url` has no unique constraint, so idempotency is select-then-insert on `url` (same pattern `user_migration` uses for `profilePicURL`).

In [2]:
def fetch_files():
    files, seen, start, limit = [], set(), 0, 100
    while True:
        r = requests.get(
            f"{CMS_BASE_URL}/api/upload/files",
            headers=HEADERS,
            params={"pagination[start]": start, "pagination[limit]": limit, "sort": "id"},
            timeout=60,
        )
        r.raise_for_status()
        body = r.json()
        batch = body["results"] if isinstance(body, dict) else body
        new = [f for f in batch if f["id"] not in seen]
        if not new:  # server ignored pagination (returned everything) or done
            return files
        files.extend(new)
        seen.update(f["id"] for f in new)
        if len(batch) < limit:
            return files
        start += limit


legacy_files = fetch_files()
print(f"fetched {len(legacy_files)} files")

fetched 9502 files


In [3]:
conn.rollback()  # clear any aborted transaction from a previous failed run

inserted = updated = skipped_no_url = 0
seen_urls = set()

with conn.cursor() as cur:
    for f in legacy_files:
        url = (f.get("url") or "").strip()
        if not url:
            skipped_no_url += 1
            continue
        if url.startswith("/"):  # relative upload path -> absolute
            url = CMS_BASE_URL + url
        if url in seen_urls:  # legacy duplicates collapse into one media row
            continue
        seen_urls.add(url)

        alt = f.get("alternativeText") or f.get("caption") or f.get("name") or None
        mime = f.get("mime")
        width, height = f.get("width"), f.get("height")

        cur.execute("SELECT id FROM media WHERE url = %s", (url,))
        row = cur.fetchone()
        if row:
            cur.execute(
                """
                UPDATE media SET mime_type = %s, alt = %s, width = %s, height = %s
                WHERE id = %s
                """,
                (mime, alt, width, height, row[0]),
            )
            updated += 1
        else:
            cur.execute(
                """
                INSERT INTO media (url, mime_type, alt, width, height)
                VALUES (%s, %s, %s, %s, %s)
                """,
                (url, mime, alt, width, height),
            )
            inserted += 1

conn.commit()
print(f"media inserted: {inserted}, updated: {updated}, skipped (no url): {skipped_no_url}")

media inserted: 0, updated: 9502, skipped (no url): 0


## 2. User Types — `user_types` → `user_types`

**This notebook is now the single source of user types** (`seed.py` no longer seeds them). Every legacy type is upserted on `slug` with its `isDefault`/`isCreator`/`isAdmin` flags; types without a slug get one generated from the name.


In [4]:
conn.rollback()  # clear any aborted transaction from a previous failed run

user_types = fetch_all("/api/user-types")
print(f"fetched {len(user_types)} user types")

skipped_types = []
with conn.cursor() as cur:
    for t in user_types:
        a = attrs(t)
        name = (a.get("name") or "").strip()
        slug = a.get("slug") or slugify(name)
        if not name or not slug:
            skipped_types.append(t["id"])
            continue
        cur.execute(
            """
            INSERT INTO user_types (name, slug, is_default, is_creator, is_admin)
            VALUES (%s, %s, %s, %s, %s)
            ON CONFLICT (slug) DO UPDATE
            SET name = EXCLUDED.name,
                is_default = EXCLUDED.is_default,
                is_creator = EXCLUDED.is_creator,
                is_admin = EXCLUDED.is_admin
            """,
            (name, slug, bool(a.get("isDefault")), bool(a.get("isCreator")), bool(a.get("isAdmin"))),
        )
conn.commit()
print(f"upserted user_types; skipped (no name): {skipped_types}")

fetched 11 user types
upserted user_types; skipped (no name): []


## 3. Companies — `companies` → `companies`

Legacy Company has `name`, `website` and an `icon` media. The icon is **kept**: its file becomes/reuses a `media` row and is linked via `companies.icon_media_id` (added by schema migration `add-company-icon`).

New schema needs a unique `slug` (generated here, de-duplicated within the run) and a `status` — set to `active` since these are existing live companies (the schema default `pending` is for new self-signups). `contact_email`/`contact_phone` stay NULL — nothing to map.

Sorted by legacy id so generated slug suffixes (`acme-2`) stay stable across re-runs.


In [5]:
conn.rollback()  # clear any aborted transaction from a previous failed run

companies = sorted(fetch_all("/api/companies", {"populate": "icon"}), key=lambda c: c["id"])
print(f"fetched {len(companies)} companies")

skipped_companies = []
used_slugs = set()


def unique_slug(base):
    base = base or "company"
    slug, n = base, 2
    while slug in used_slugs:
        slug = f"{base}-{n}"
        n += 1
    used_slugs.add(slug)
    return slug


with conn.cursor() as cur:
    for c in companies:
        a = attrs(c)
        name = (a.get("name") or "").strip()
        if not name:
            skipped_companies.append(c["id"])
            continue
        slug = unique_slug(slugify(name))

        # icon -> media row (reuses the row step 1 created for the same url)
        icon, icon_media_id = rel(a.get("icon")), None
        if icon and icon.get("url"):
            url = icon["url"].strip()
            if url.startswith("/"):
                url = CMS_BASE_URL + url
            cur.execute("SELECT id FROM media WHERE url = %s", (url,))
            row = cur.fetchone()
            if not row:
                cur.execute(
                    """
                    INSERT INTO media (url, mime_type, alt, width, height)
                    VALUES (%s, %s, %s, %s, %s) RETURNING id
                    """,
                    (url, icon.get("mime"), icon.get("alternativeText") or icon.get("name"),
                     icon.get("width"), icon.get("height")),
                )
                row = cur.fetchone()
            icon_media_id = row[0]

        cur.execute(
            """
            INSERT INTO companies (name, slug, website, icon_media_id, status)
            VALUES (%s, %s, %s, %s, 'active')
            ON CONFLICT (slug) DO UPDATE
            SET name = EXCLUDED.name,
                website = EXCLUDED.website,
                icon_media_id = EXCLUDED.icon_media_id
            """,
            (name, slug, (a.get("website") or "").strip() or None, icon_media_id),
        )
conn.commit()
print(f"upserted companies; skipped (no name): {skipped_companies}")


fetched 309 companies
upserted companies; skipped (no name): [36, 45, 47, 48, 49, 50, 51, 53, 54, 55, 56, 57, 58, 59, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 94, 95, 99, 100, 101, 102, 103, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 132, 133, 134, 136, 137, 139, 143, 144, 145, 146, 147, 148, 149, 150, 151, 153, 154, 155, 156, 158, 161, 163, 170]


## 4. Verify

In [6]:
with conn.cursor() as cur:
    for t in ("media", "user_types", "companies"):
        cur.execute(f"SELECT COUNT(*) FROM {t}")
        print(f"{t:12}: {cur.fetchone()[0]}")
    cur.execute("SELECT slug, is_default, is_creator, is_admin FROM user_types ORDER BY id")
    for slug, d, cr, ad in cur.fetchall():
        print(f"  user_type {slug:15} default={d} creator={cr} admin={ad}")
conn.close()

media       : 9740
user_types  : 11
companies   : 210
  user_type regular         default=True creator=False admin=False
  user_type super-admin     default=False creator=True admin=True
  user_type story-teller    default=False creator=True admin=False
  user_type tour-operators  default=False creator=True admin=False
  user_type hotels          default=False creator=True admin=False
  user_type admin-1         default=False creator=False admin=True
  user_type editorial-admin default=False creator=True admin=True
  user_type editor-in-chief default=False creator=True admin=True
  user_type destination-expert default=False creator=False admin=False
  user_type concierge-member default=False creator=False admin=False
  user_type fnd-expert      default=False creator=True admin=True
